# Support vector machines
Suport vector machines is a supervised ML algorithm used for regression and classification tasks. Usually defined as linear classifiers to find the ideal decision boundary that maximizes the margin among support vectors.

## Core concepts
- **Support vector:** A vector defined by the points closest to the decision boundary in a class.
- **Margin:** Distance between support vectors of the classes.
- **Decision boundary:** The hyperplane that separates both classes.

### Kernel functions
Many times isn't possible to find a linear hyperplane that separates the classes, in these cases is necessary to add another dimension to find a linear decision boundary. However, adding another dimension is computationally expensive. So the SVM builds a "similitude matrix" evaluating a kernel function $K(\vec{x_{i}}, \vec{x_{j}})$ that maps the similarity between each couple of data instances. For inference, just evaluates the similarity between the data instance and each support vector. Common kernels are:
- **Linear:** $K(\vec{x}, \vec{z})=\vec{x} \cdot \vec{z}$
- **Polinomial:** $K(\vec{x}, \vec{z})=(\vec{x} \cdot \vec{z} + c)^{d}$, where $d$ is the degree and $c$ the individual term.
- **Radial (RBF):** $K(\vec{x}, \vec{z})=exp(-\gamma || \vec{x} - \vec{z}||^{2})$, where $\gamma$ is a defined parameter.
- **Sigmoid:** $K(\vec{x}, \vec{z})=tanh(\gamma\vec{x} \cdot \vec{z} + c)$, where $c$ and $\alpha$ are parameters.

## How does it works?
## Classification
In classification tasks, the goal is to find the hyperplane that maximizes the margin between the best support vectors. As we mentioned, sometimes it's necessary to apply some transformings to our feature space in an effort for finding good support vectors. Keep in mind that this is a quadratic programming problem, so, on training step it's computationally expensive. At inference, it's just follow this formula:

<center>
    $\bar{y}(x) = sign(\sum_{i \in SV} y_{i} \alpha_{i} K(x_{i}, x) + b)$
</center

where:
- $x$ is the instance to be classified.
- $x_{i}$ is the support vector $i$.
- $y_{i}$ is the class label of support vector $x_{i}$, represented as $1$ or $-1$ (positive and negative classes respectively).
- $\alpha_{i}$ is the Langrage multiplier of support vector $i$.
- $K(x_{i}, x)$ is the kernel function measuring similarty between $x$ and support vector $x_{i}$
- $b$ a bias coefficient.
- $sign(z)$ is a function that returns $1$ if $z$ is a positive value and $-1$ in the opposite case.

<center>
    <img width="400px" height="200px" src="https://cdn.learnopencv.com/wp-content/uploads/2018/07/04095847/support-vectors-and-maximum-margin.png" alt="svm">
</center>

  
It's just necessary to perform some calculations using the support vectors (2), so it's very efficient. However, as you guess SVM doesn't support multilabel tasks because is a binary classifier. To issue this, libraries like Scikit implements 2 strategies:
1. **One vs one (ovo):** Trains an SVM for each couple class combination. For instance, in a case where we got $C_{1}, C_{2}, C_{3}$ classes, there would be $(C_{1}, C_{2}), (C_{1}, C_{3}), (C_{2}, C_{3})$ classifiers and for inference, works like a voting system that chooses the most voted class.
2. **One vr rest (ovr):** Trains an SVM per class, the goal is to train a classifier that specializes in identifying a single class (the negative class is considered as the rest). For inference evaluates the decision function (scores the distance of a sample from the decision boundary for each class) of each classifier and selects the one with the highest score.

In [1]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = GaussianNB()
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

Accuracy: 82.000000 %
Weighted F1-Score: 81.787420 %


## Regression
As classification tasks we need to find a decision boundary, but in this case it tries to fit the target function as plane as possible. Unlike linear regression, SVM doesn't cares about minimizing individual errors, it cares about global error. To achieve this goal, we define the concept of "epsilon tube". The epsilon tube is a margin around the decision boundary defined by $\epsilon$ hyperparameter, that represents a threshold that tells us "hey, data points inside this region are good enough", so we consider that $error = 0$. Is supposed that points outside epsilon tube have $error > 0$, so as they are out of the margin (epsilon bound) they are support vectors.

<center>
    <img width="600px" height="400px" src="https://media.datacamp.com/cms/61aca46ac2119ca80479f0a90c705cd8.png" alt="epsilon-tube">
</center>

Training process consists on finding the decision boundary that captures data points where $error \rightarrow 0$ into the epsilon tube to use the support vectors that best represents target function behavior with least error possible. As easy as find the support vectors, Lagrange multipliers and intercept of the following function:

<center>
    $\bar{y}(x) = \sum_{i \in SV} (\alpha_{i} - \alpha_{i}^{*}) K(x_{i}, x) + b$
</center>

  
where:
- $x$ is the instance to be classified.
- $x_{i}$ is the support vector $i$.
- $\alpha_{i}, \alpha_{i}^{*}$ are the Langrage multipliers of support vector $i$ (the difference is often calculated as a single coefficient by ML frameworks).
- $K(x_{i}, x)$ is the kernel function measuring similarty between $x$ and support vector $x_{i}$
- $b$ a bias coefficient.

In [2]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = SVR(C=0.5, kernel="sigmoid", epsilon=0.5)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

Mean squared error: 6978064896.919157
Mean absolute error: 27634.634234


## Most used hyperparameters
- **`c`:** Regularization parameter (a positive value), its strenght is inversely proportional to $C$. Only available in `SVC`, `SVR`, `LinearSVC` and `LinearSVR` classes.
- **`nu`:** An upper bound to the fraction of margin errors and a lower bound of the fraction of support vectors. Only available in `NuSVC` and `NuSVR` classes.
- **`kernel`:** Specifies the kernel to be used.
- **`degree`:** Degree of the polynomial kernel function.
- **`gamma`:** $\gamma$ coefficient used in some kernels.
- **`coef0`:** $c$ independent coefficient.
- **`tol`:** Tolerance for stopping criterion.
- **`epsilon`:** $\epsilon$ width of the epsilon tube (only used in regression models).

**Note:** According to Scikit documentation, `NuSVC` and `SVC` (the same applies to the regression classes) have the same mathematical definition, but instead $C$ we have $Nu$ as explained.

## Advantages
- Performs well in high dimensional datasets.
- Easy to explain method.
- Works fast at inference.

## Disadvantages
- Finding ideal support vectors is a quadratic programming problem, so it's computationally expensive.
- For multiclass classification tasks, must train more than 1 classifier.
- Performs bad when there are more data instances than features.
- Requires data scaling to a defined interval like $[0,1]$ or $[-1,1]$ or standarizing data.
- Doesn't calculates probabilities, when does, often differs form model outcomes.